# 03 · RAG queries

Ask comparison questions against the vector store and save the structured answers to `data/processed/rag_results.json` for analysis.

Requires `OPENAI_API_KEY` and the database from `02_build_vectorstore.ipynb`.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when running from notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

In [ ]:
import json

from src.config import CHAT_MODEL, RESULTS_FILE, require_api_key
from src.rag_chain import build_rag
from src.utils import parse_json_answer, pretty_print_sources
from src.vectorstore import load_vectorstore, vectorstore_exists

require_api_key()
assert vectorstore_exists(), "Run 02_build_vectorstore.ipynb first."

qa = build_rag(load_vectorstore())
print("Chat model:", CHAT_MODEL)

In [ ]:
questions = [
    "Compare MaryAnne and Sue's environmental exposures with scientific evidence.",
    "Compare MaryAnne and Sue's lifestyle factors (diet, alcohol, weight, physical activity) with the evidence.",
    "Compare the psychosocial stressors in MaryAnne and Sue's lives with the evidence on stress and breast cancer.",
    "Which early-life (birth to age 10) factors differ between MaryAnne and Sue, and what does the evidence say about them?",
]

In [ ]:
results = []
for q in questions:
    out = qa.invoke({"query": q})
    parsed = parse_json_answer(out["result"])
    results.append(
        {
            "question": q,
            "answer": parsed,
            "raw_answer": out["result"],
            "sources": [d.metadata for d in out["source_documents"]],
        }
    )
    print(f"\n=== {q}")
    print(json.dumps(parsed, indent=2) if parsed else out["result"])
    pretty_print_sources(out["source_documents"])

In [ ]:
RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
RESULTS_FILE.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved {len(results)} results to {RESULTS_FILE.relative_to(ROOT)}")